# Tutorial 2: Working with EXFOR (Experimental Nuclear Reaction Data)

## Overview

EXFOR (Exchange Format) is the international database of experimental nuclear reaction data maintained by the IAEA. Unlike ENDF (which contains evaluated data), EXFOR contains raw experimental measurements from publications.

### What you'll learn:
- How to download EXFOR data from the IAEA website
- How to load and analyze experimental measurements
- How to visualize experimental data with uncertainties
- How to compare experimental data with evaluated data (from Tutorial 1)
- How to prepare experimental data for machine learning

### Prerequisites:
```bash
pip install requests matplotlib numpy pandas scipy
```

### ⚠️ IMPORTANT: This tutorial requires REAL experimental data!
You must download actual EXFOR data from the IAEA website before running this notebook.

## 1. Understanding EXFOR

### Key Concepts:

- **Experimental Data**: Direct measurements from experiments, not evaluations
- **Uncertainties**: Experimental data includes measurement uncertainties
- **Entry Numbers**: Each experiment has a unique EXFOR entry number
- **Subentry**: Individual measurements within an entry
- **REACTION**: Specifies what was measured (e.g., U-235(n,f) = neutron-induced fission of U-235)

### Why EXFOR is important:
- Source data for evaluations
- Validation of evaluated libraries
- Uncertainty quantification
- Machine learning training data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

# Create data directory
data_dir = Path('../data/exfor')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"Data directory: {data_dir.absolute()}")

## 2. 📥 DOWNLOADING REAL EXFOR DATA

### Step-by-Step Instructions:

#### For U-235 Fission Cross-Section:

1. **Visit the IAEA EXFOR Database:**
   - Go to: https://www-nds.iaea.org/exfor/

2. **Search for Data:**
   - Click on "Retrieval" in the menu
   - Select "Data Retrieval" → "Cross Section"
   - Target: Enter `92-U-235`
   - Reaction: Enter `(N,F)` (neutron-induced fission)
   - Energy Range: `1.0E-2 eV` to `2.0E+7 eV`
   - Click "Search"

3. **Select Experiments:**
   - You'll see a list of available experiments
   - Choose 3-5 different experiments (look for different years/authors)
   - Select experiments that cover different energy ranges

4. **Download Data:**
   - Select "Download" → "CSV format"
   - Save each experiment as: `exfor_experiment_1.csv`, `exfor_experiment_2.csv`, etc.
   - Save all files to: `../data/exfor/`

#### Alternative: JANIS Interface (Easier!)

1. Go to: https://www.oecd-nea.org/janisweb/
2. Click "Search" → "Experimental Data"
3. Select U-235, reaction (n,f)
4. Choose multiple experiments from the list
5. Export each to CSV format

### 📋 Required Files:
- At least 3 EXFOR experimental data files (CSV format)
- Files should contain columns: Energy, CrossSection, Uncertainty (or similar)
- Save all files to `../data/exfor/` directory

## 3. Loading Real EXFOR Data

Let's load the experimental data you downloaded. The code will check that you have real EXFOR files.

In [ ]:
# Check for EXFOR CSV files
exfor_files = sorted(data_dir.glob('exfor_*.csv'))

if len(exfor_files) == 0:
    print("❌ ERROR: No EXFOR data files found!")
    print(f"\nExpected directory: {data_dir.absolute()}")
    print("\n📥 PLEASE DOWNLOAD REAL EXFOR DATA:")
    print("\n   Method 1 - IAEA EXFOR Database:")
    print("   1. Go to https://www-nds.iaea.org/exfor/")
    print("   2. Search: Target='92-U-235', Reaction='(N,F)'")
    print("   3. Select 3-5 different experiments")
    print("   4. Download each as CSV format")
    print("   5. Save as: exfor_experiment_1.csv, exfor_experiment_2.csv, etc.")
    print(f"   6. Save to directory: {data_dir.absolute()}")
    print("\n   Method 2 - JANIS Interface (Recommended):")
    print("   1. Go to https://www.oecd-nea.org/janisweb/")
    print("   2. Search → Experimental Data")
    print("   3. Select U-235, (n,f) reaction")
    print("   4. Choose multiple experiments from list")
    print("   5. Export each to CSV")
    print(f"   6. Save to directory: {data_dir.absolute()}")
    print("\n⚠️ This tutorial REQUIRES real experimental data!")
    raise FileNotFoundError(f"No EXFOR data files found in {data_dir}")

print(f"✓ Found {len(exfor_files)} EXFOR data files:")
for f in exfor_files:
    print(f"  - {f.name}")

## 4. Parse and Load Experimental Data

EXFOR CSV files can have different column formats. We'll try to intelligently parse them.

In [ ]:
def parse_exfor_csv(file_path):
    """
    Parse EXFOR CSV file and standardize column names.
    
    Different EXFOR exports may have different column names.
    This function tries to identify and standardize them.
    """
    try:
        # Try reading with different settings
        df = pd.read_csv(file_path, comment='#', skipinitialspace=True)
        
        # Identify energy column (case-insensitive)
        energy_cols = [col for col in df.columns if 'energ' in col.lower() or 'en' == col.lower()]
        xs_cols = [col for col in df.columns if 'data' in col.lower() or 'xs' in col.lower() or 'cross' in col.lower()]
        err_cols = [col for col in df.columns if 'err' in col.lower() or 'unc' in col.lower() or 'd' in col.lower()]
        
        if not energy_cols or not xs_cols:
            print(f"⚠️ Warning: Could not identify columns in {file_path.name}")
            print(f"   Available columns: {df.columns.tolist()}")
            return None
        
        # Create standardized dataframe
        result = pd.DataFrame()
        result['Energy_eV'] = pd.to_numeric(df[energy_cols[0]], errors='coerce')
        result['CrossSection_barns'] = pd.to_numeric(df[xs_cols[0]], errors='coerce')
        
        # Try to get uncertainty
        if err_cols:
            result['Uncertainty_barns'] = pd.to_numeric(df[err_cols[0]], errors='coerce')
        else:
            # If no uncertainty column, assume 5% uncertainty
            print(f"   Note: No uncertainty column found, assuming 5% uncertainty")
            result['Uncertainty_barns'] = result['CrossSection_barns'] * 0.05
        
        # Remove any rows with NaN values
        result = result.dropna()
        
        # Add metadata
        result['Source_File'] = file_path.name
        
        return result
        
    except Exception as e:
        print(f"❌ Error parsing {file_path.name}: {e}")
        return None

# Load all EXFOR experiments
experiments = []
for exfor_file in exfor_files:
    print(f"\nLoading: {exfor_file.name}")
    df = parse_exfor_csv(exfor_file)
    if df is not None and len(df) > 0:
        experiments.append(df)
        print(f"  ✓ Loaded {len(df)} data points")
        print(f"    Energy range: {df['Energy_eV'].min():.2e} - {df['Energy_eV'].max():.2e} eV")
        print(f"    XS range: {df['CrossSection_barns'].min():.2f} - {df['CrossSection_barns'].max():.2f} barns")

if len(experiments) == 0:
    print("\n❌ ERROR: Failed to load any EXFOR data!")
    print("   Please check that your CSV files are formatted correctly.")
    raise ValueError("No valid EXFOR data loaded")

print(f"\n✓ Successfully loaded {len(experiments)} experiments [REAL DATA]")

## 5. Visualizing Experimental Data with Uncertainties [REAL DATA]

A key feature of experimental data is the uncertainty bars, which represent measurement errors.

In [ ]:
plt.figure(figsize=(14, 8))

colors = plt.cm.Set1(np.linspace(0, 1, len(experiments)))
markers = ['o', 's', '^', 'v', 'D', 'p', '*', 'h']

for i, exp in enumerate(experiments):
    marker = markers[i % len(markers)]
    plt.errorbar(
        exp['Energy_eV'], 
        exp['CrossSection_barns'], 
        yerr=exp['Uncertainty_barns'],
        fmt=marker,
        color=colors[i],
        label=f"Experiment {i+1} ({len(exp)} pts)",
        markersize=6,
        capsize=3,
        alpha=0.7
    )

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
plt.ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
plt.title('U-235 Fission Cross-Section: REAL Experimental Data from EXFOR', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3, which='both')

# Add watermark
plt.text(0.98, 0.02, '[REAL DATA]', 
         transform=plt.gca().transAxes,
         fontsize=12, color='red', fontweight='bold',
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(data_dir / 'exfor_u235_fission_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Plot saved: {data_dir / 'exfor_u235_fission_REAL.png'}")

## 6. Comparing EXFOR with ENDF Evaluation [REAL DATA]

Let's overlay experimental EXFOR data with the evaluated ENDF data from Tutorial 1.

In [ ]:
# Load ENDF data from Tutorial 1
endf_dir = Path('../data/endf')
endf_fission_file = endf_dir / 'u235_fission_endf8.csv'

if not endf_fission_file.exists():
    print("⚠️ WARNING: ENDF evaluation data not found!")
    print(f"   Expected file: {endf_fission_file}")
    print("\n   Please complete Tutorial 1 first to download ENDF data.")
    print("   For now, we'll skip the EXFOR vs ENDF comparison.")
    df_endf = None
else:
    try:
        df_endf = pd.read_csv(endf_fission_file)
        print(f"✓ Loaded ENDF evaluation: {len(df_endf)} points [REAL DATA]")
    except Exception as e:
        print(f"❌ Error loading ENDF data: {e}")
        df_endf = None

In [ ]:
# Plot comparison if ENDF data available
if df_endf is not None:
    plt.figure(figsize=(14, 8))
    
    # Plot ENDF evaluation as a line
    plt.plot(df_endf['Energy_eV'], df_endf['CrossSection_barns'], 
             'k-', linewidth=2.5, label='ENDF/B-VIII.0 Evaluation', alpha=0.7, zorder=1)
    
    # Plot experimental data points
    for i, exp in enumerate(experiments):
        marker = markers[i % len(markers)]
        plt.errorbar(
            exp['Energy_eV'], 
            exp['CrossSection_barns'], 
            yerr=exp['Uncertainty_barns'],
            fmt=marker,
            color=colors[i],
            label=f"EXFOR Exp. {i+1}",
            markersize=7,
            capsize=3,
            alpha=0.8,
            zorder=2
        )
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
    plt.ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
    plt.title('U-235 Fission: ENDF Evaluation vs EXFOR Experimental Data', 
              fontsize=16, fontweight='bold', pad=20)
    plt.legend(fontsize=10, loc='best', framealpha=0.9)
    plt.grid(True, alpha=0.3, which='both')
    
    # Add watermark
    plt.text(0.98, 0.02, '[REAL DATA]', 
             transform=plt.gca().transAxes,
             fontsize=12, color='red', fontweight='bold',
             ha='right', va='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(data_dir / 'exfor_vs_endf_REAL.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Comparison plot saved: {data_dir / 'exfor_vs_endf_REAL.png'}")
else:
    print("⏭️ Skipping EXFOR vs ENDF comparison (ENDF data not available)")

## 7. Statistical Analysis of Experimental Data [REAL DATA]

In [ ]:
# Combine all experimental data into a single DataFrame
df_all_exfor = pd.concat(experiments, ignore_index=True)

# Add relative uncertainty column
df_all_exfor['Relative_Uncertainty_%'] = (df_all_exfor['Uncertainty_barns'] / 
                                            df_all_exfor['CrossSection_barns']) * 100

print("="*60)
print("EXFOR REAL DATA SUMMARY")
print("="*60)
print(f"\nTotal experiments: {len(experiments)}")
print(f"Total data points: {len(df_all_exfor)}")
print(f"\nEnergy range: {df_all_exfor['Energy_eV'].min():.2e} - {df_all_exfor['Energy_eV'].max():.2e} eV")
print(f"XS range: {df_all_exfor['CrossSection_barns'].min():.3f} - {df_all_exfor['CrossSection_barns'].max():.3f} barns")
print(f"\nMean relative uncertainty: {df_all_exfor['Relative_Uncertainty_%'].mean():.2f}%")
print(f"Median relative uncertainty: {df_all_exfor['Relative_Uncertainty_%'].median():.2f}%")

print("\nDetailed Statistics:")
print(df_all_exfor[['Energy_eV', 'CrossSection_barns', 'Uncertainty_barns', 'Relative_Uncertainty_%']].describe())

# Save combined data to CSV
csv_file = data_dir / 'exfor_u235_fission_combined_REAL.csv'
df_all_exfor.to_csv(csv_file, index=False)
print(f"\n✓ Combined data saved: {csv_file}")

## 8. Uncertainty Analysis [REAL DATA]

Let's analyze how uncertainties vary across different energy ranges.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Absolute uncertainties
for i, exp in enumerate(experiments):
    marker = markers[i % len(markers)]
    ax1.loglog(
        exp['Energy_eV'], 
        exp['Uncertainty_barns'],
        marker,
        color=colors[i],
        label=f"Experiment {i+1}",
        markersize=8
    )

ax1.set_xlabel('Neutron Energy (eV)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Absolute Uncertainty (barns)', fontsize=12, fontweight='bold')
ax1.set_title('Absolute Uncertainties in REAL Experimental Data', 
              fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, which='both')

# Plot 2: Relative uncertainties
for i, exp in enumerate(experiments):
    marker = markers[i % len(markers)]
    ax2.semilogx(
        exp['Energy_eV'], 
        exp['Relative_Uncertainty_%'],
        marker,
        color=colors[i],
        label=f"Experiment {i+1}",
        markersize=8
    )

ax2.set_xlabel('Neutron Energy (eV)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Relative Uncertainty (%)', fontsize=12, fontweight='bold')
ax2.set_title('Relative Uncertainties in REAL Experimental Data', 
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / 'exfor_uncertainties_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Uncertainty analysis saved: {data_dir / 'exfor_uncertainties_REAL.png'}")

## 9. Residual Analysis: EXFOR vs ENDF [REAL DATA]

Calculate how well the ENDF evaluation matches experimental measurements.

In [ ]:
if df_endf is not None:
    # For each EXFOR point, find closest ENDF value and calculate residual
    residuals = []
    normalized_residuals = []
    
    for exp in experiments:
        for idx, row in exp.iterrows():
            energy = row['Energy_eV']
            xs_exp = row['CrossSection_barns']
            unc_exp = row['Uncertainty_barns']
            
            # Find closest ENDF energy point
            closest_idx = (np.abs(df_endf['Energy_eV'] - energy)).argmin()
            xs_endf = df_endf.iloc[closest_idx]['CrossSection_barns']
            
            # Calculate residuals
            residual = xs_exp - xs_endf
            normalized_residual = residual / unc_exp  # In units of σ
            
            residuals.append(residual)
            normalized_residuals.append(normalized_residual)
    
    residuals = np.array(residuals)
    normalized_residuals = np.array(normalized_residuals)
    
    # Plot normalized residuals
    plt.figure(figsize=(12, 6))
    plt.hist(normalized_residuals, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Perfect agreement')
    plt.axvline(np.mean(normalized_residuals), color='blue', linestyle='--', linewidth=2, 
                label=f'Mean = {np.mean(normalized_residuals):.2f} σ')
    plt.axvline(-3, color='orange', linestyle=':', linewidth=1.5, alpha=0.5, label='±3σ bounds')
    plt.axvline(3, color='orange', linestyle=':', linewidth=1.5, alpha=0.5)
    
    plt.xlabel('Normalized Residual (σ)', fontsize=12, fontweight='bold')
    plt.ylabel('Frequency', fontsize=12, fontweight='bold')
    plt.title('Distribution of Residuals: REAL EXFOR vs ENDF Evaluation', 
              fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    
    # Add watermark
    plt.text(0.98, 0.95, '[REAL DATA]', 
             transform=plt.gca().transAxes,
             fontsize=12, color='red', fontweight='bold',
             ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(data_dir / 'residuals_REAL.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print("="*60)
    print("RESIDUAL ANALYSIS [REAL DATA]")
    print("="*60)
    print(f"Mean residual: {np.mean(normalized_residuals):.2f} σ")
    print(f"Std deviation: {np.std(normalized_residuals):.2f} σ")
    print(f"Median residual: {np.median(normalized_residuals):.2f} σ")
    print(f"\nPoints within ±1σ: {np.sum(np.abs(normalized_residuals) <= 1)} ({100*np.sum(np.abs(normalized_residuals) <= 1)/len(normalized_residuals):.1f}%)")
    print(f"Points within ±2σ: {np.sum(np.abs(normalized_residuals) <= 2)} ({100*np.sum(np.abs(normalized_residuals) <= 2)/len(normalized_residuals):.1f}%)")
    print(f"Points within ±3σ: {np.sum(np.abs(normalized_residuals) <= 3)} ({100*np.sum(np.abs(normalized_residuals) <= 3)/len(normalized_residuals):.1f}%)")
    print(f"\nOutliers (>3σ): {np.sum(np.abs(normalized_residuals) > 3)}")
    
    print(f"\n✓ Residual analysis saved: {data_dir / 'residuals_REAL.png'}")
else:
    print("⏭️ Skipping residual analysis (ENDF data not available)")

## 10. Machine Learning Applications

EXFOR data is valuable for machine learning because:

1. **Training Data**: Use experimental measurements to train ML models
2. **Validation**: Compare ML predictions with independent measurements
3. **Outlier Detection**: Identify potentially problematic measurements
4. **Uncertainty Quantification**: Learn patterns in experimental uncertainties
5. **Data Consistency**: Check consistency between different experiments

### Preparing Data for ML

In [ ]:
# Create ML-ready dataset
ml_data = df_all_exfor.copy()

# Add log-scale features (useful for ML)
ml_data['Log10_Energy'] = np.log10(ml_data['Energy_eV'])
ml_data['Log10_CrossSection'] = np.log10(ml_data['CrossSection_barns'])
ml_data['Log10_Uncertainty'] = np.log10(ml_data['Uncertainty_barns'])

# Add energy bins for classification
def classify_energy_region(energy):
    if energy < 1:
        return 'Thermal'
    elif energy < 1e3:
        return 'Epithermal'
    elif energy < 1e5:
        return 'Resonance'
    else:
        return 'Fast'

ml_data['Energy_Region'] = ml_data['Energy_eV'].apply(classify_energy_region)

# Save ML-ready dataset
ml_file = data_dir / 'exfor_ML_dataset_REAL.csv'
ml_data.to_csv(ml_file, index=False)

print("="*60)
print("ML DATASET PREPARED [REAL DATA]")
print("="*60)
print(f"\nTotal samples: {len(ml_data)}")
print(f"\nFeatures included:")
print(f"  - Energy (eV and log10)")
print(f"  - Cross-section (barns and log10)")
print(f"  - Uncertainty (barns and log10)")
print(f"  - Relative uncertainty (%)")
print(f"  - Energy region classification")
print(f"  - Source file metadata")
print(f"\nData points by energy region:")
print(ml_data['Energy_Region'].value_counts())
print(f"\n✓ ML dataset saved: {ml_file}")

## 11. Key Takeaways

### What We Learned:

1. **EXFOR contains REAL experimental data** - Raw measurements with uncertainties
2. **Multiple experiments exist** for the same reaction - Provides validation
3. **Uncertainties are crucial** - They quantify measurement quality
4. **Scatter between experiments** is normal - Due to different methods, facilities, etc.
5. **EXFOR is the foundation** for evaluated libraries like ENDF
6. **Residual analysis** helps validate evaluations against experiments
7. **Experimental data is essential** for training and validating ML models

### Important for Machine Learning:

- Use REAL experimental data for training (not simulated!)
- Account for measurement uncertainties in your models
- Compare predictions with multiple independent experiments
- Understand the physics behind different energy regions
- Use residual analysis to identify model weaknesses

## Next Steps

- Download EXFOR data for other isotopes (Pu-239, Fe-56, etc.)
- Compare different reaction types (capture, elastic scattering, etc.)
- Analyze systematic differences between facilities
- Explore Tutorial 3 (TENDL) and Tutorial 4 (JANIS multi-library comparison)
- Train ML models on this REAL experimental data

## Resources

- **EXFOR Database:** https://www-nds.iaea.org/exfor/
- **JANIS Interface:** https://www.oecd-nea.org/janisweb/
- **x4i3 Python Package:** https://github.com/afedynitch/x4i3
- **IAEA Nuclear Data Services:** https://www-nds.iaea.org/
- **EXFOR Manual:** https://www-nds.iaea.org/nrdc/exfor-man/